# 08 modelos 18 y 19



## Modelo 18 — Validación con montos parecidos a los del test

**Qué probamos.** Cambiamos la forma de validar. Utilizamos solicitudes de clientes que no aparecían antes en el entrenamiento y evaluamos rangos de monto presentes en enero de 2026, con presupuesto proporcional a la demanda del conjunto evaluado.

**Qué pasó.** La regresión actual obtuvo un ROI de 25,026% en el escenario de rango completo del test. Entrenar solo con montos dentro de ese rango llegó a 25,189%; retirar monto y meses observados obtuvo 25,011%. Al acotar al percentil 95 del test, las diferencias también fueron pequeñas.

**Qué decidimos.** El cambio de validación nos sirvió para detectar diferencias de población, pero no mostraba una mejora clara por sí solo. Lo usamos como punto de partida para las siguientes pruebas.

In [ ]:

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

# trabajamos con copias para no modificar lo anterior
m18_train = train_fe.copy()
m18_test = test_fe.copy()

m18_train["Fecha_Solicitud"] = pd.to_datetime(
    m18_train["Fecha_Solicitud"]
)
m18_test["Fecha_Solicitud"] = pd.to_datetime(
    m18_test["Fecha_Solicitud"]
)

# definimos las caracteristicas que vamos a comparar
m18_columnas_base = list(columnas_m16)

m18_columnas_sin = [
    c for c in m18_columnas_base
    if c not in ("Monto_Solicitado", "Meses_Observados")
]

# comprobamos que los datos esten completos
for df in (m18_train, m18_test):
    assert df["ID_Solicitud"].is_unique
    assert df[m18_columnas_base].notna().all().all()

# analizamos los montos reales del test
m18_max_test = float(m18_test["Monto_Solicitado"].max())
m18_p95_test = float(
    m18_test["Monto_Solicitado"].quantile(0.95)
)

m18_demanda_test = float(
    m18_test["Monto_Solicitado"].sum()
)

m18_fraccion_capital = min(
    1.0, 2_000_000 / m18_demanda_test
)

# revisamos las diferencias entre train y test
print("Solicitudes train/test:", len(m18_train), len(m18_test))

print(
    "Usuarios de test presentes en train:",
    m18_test["ID_Usuario"].isin(
        m18_train["ID_Usuario"]
    ).sum()
)

print(
    "Monto mediano train/test:",
    m18_train["Monto_Solicitado"].median(),
    m18_test["Monto_Solicitado"].median()
)

print("Monto maximo de test:", m18_max_test)
print("Percentil 95 de test:", m18_p95_test)

print(
    "Proporcion del capital disponible:",
    round(m18_fraccion_capital * 100, 2), "%"
)


In [ ]:

# construimos la regresion con las variables elegidas
def crear_logistica_m18(columnas):
    numeros = [
        c for c in columnas
        if c != "Motivo_Prestamo"
    ]

    prep = ColumnTransformer([
        ("numericas", StandardScaler(), numeros),
        ("motivo", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ), ["Motivo_Prestamo"])
    ])

    return Pipeline([
        ("preprocesador", prep),
        ("modelo", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])


# evaluamos una cartera con el presupuesto indicado
def evaluar_m18(df, probabilidades, presupuesto):
    cartera = df[
        ["ID_Solicitud", "Monto_Solicitado",
         "Tasa_Interes", "Estado_Final"]
    ].copy().reset_index(drop=True)

    cartera["Prob_Pago"] = np.asarray(probabilidades)

    # calculamos el valor esperado por dolar
    cartera["EV_Dolar"] = (
        cartera["Prob_Pago"] * (1 + cartera["Tasa_Interes"]) - 1
    )

    cartera = cartera.sort_values(
        ["EV_Dolar", "ID_Solicitud"],
        ascending=[False, True]
    ).reset_index(drop=True)

    # seleccionamos prestamos sin superar el presupuesto
    capital = 0.0
    seleccionados = []

    for fila in cartera.itertuples():
        if fila.EV_Dolar <= 0:
            continue

        if capital + fila.Monto_Solicitado <= presupuesto:
            seleccionados.append(fila.Index)
            capital += fila.Monto_Solicitado

    aprobados = cartera.iloc[seleccionados]

    # calculamos la ganancia real de los aprobados
    ganancia = np.where(
        aprobados["Estado_Final"] == 1,
        aprobados["Monto_Solicitado"] * aprobados["Tasa_Interes"],
        -aprobados["Monto_Solicitado"]
    ).sum()

    return {
        "Candidatos": len(cartera),
        "Monto_Mediano": float(
            cartera["Monto_Solicitado"].median()
        ),
        "Presupuesto": float(presupuesto),
        "Capital": float(capital),
        "Uso_Presupuesto_%": (
            float(100 * capital / presupuesto)
            if presupuesto else 0.0
        ),
        "Ganancia": float(ganancia),
        "ROI_%": (
            float(100 * ganancia / capital)
            if capital else 0.0
        ),
        "Aprobados": len(aprobados),
        "Defaults": int(
            (aprobados["Estado_Final"] == 0).sum()
        ),
        "Sesgo_Prob": float(
            cartera["Prob_Pago"].mean()
            - cartera["Estado_Final"].mean()
        ),
        "Sesgo_Aprobados": (
            float(
                aprobados["Prob_Pago"].mean()
                - aprobados["Estado_Final"].mean()
            ) if len(aprobados) else np.nan
        )
    }


In [ ]:

# definimos las tres versiones
m18_modelos = {
    "Actual": (m18_columnas_base, False),
    "Actual entrenado en montos test": (m18_columnas_base, True),
    "Sin monto ni meses observados": (m18_columnas_sin, False),
}

# usamos dos rangos de montos para comprobar estabilidad
m18_segmentos = {
    "Rango test": m18_max_test,
    "Hasta P95 test": m18_p95_test
}

m18_resultados = []
m18_control = []

# entrenamos con meses anteriores y validamos usuarios nuevos
for mes in ["2025-10", "2025-11", "2025-12"]:
    inicio = pd.Timestamp(mes + "-01")
    fin = inicio + pd.offsets.MonthBegin(1)

    datos_anteriores = m18_train[
        m18_train["Fecha_Solicitud"] < inicio
    ]

    datos_mes = m18_train[
        (m18_train["Fecha_Solicitud"] >= inicio) &
        (m18_train["Fecha_Solicitud"] < fin)
    ]

    usuarios_vistos = set(datos_anteriores["ID_Usuario"])

    datos_nuevos = datos_mes[
        ~datos_mes["ID_Usuario"].isin(usuarios_vistos)
    ]

    print(
        f"{mes}: {len(datos_mes)} solicitudes, "
        f"{len(datos_nuevos)} de usuarios no vistos"
    )

    for nombre, (columnas, solo_montos_test) in m18_modelos.items():
        df_tr = datos_anteriores

        # una version se entrena solo con montos del rango de test
        if solo_montos_test:
            df_tr = df_tr[
                df_tr["Monto_Solicitado"] <= m18_max_test
            ]

        assert len(df_tr) > 0

        modelo = crear_logistica_m18(columnas)

        modelo.fit(
            df_tr[columnas],
            df_tr["Estado_Final"]
        )

        # comprobamos que la regresion actual reproduzca la prueba anterior
        if nombre == "Actual":
            antiguo = datos_mes[
                datos_mes["Monto_Solicitado"] <= 10900
            ]

            p_antiguo = modelo.predict_proba(
                antiguo[columnas]
            )[:, 1]

            met_antiguo = evaluar_m18(
                antiguo, p_antiguo, 2_000_000
            )

            m18_control.append({
                "Mes": mes,
                "Ganancia": met_antiguo["Ganancia"]
            })

        # evaluamos con montos similares a los de kaggle
        for segmento, tope in m18_segmentos.items():
            df_val = datos_nuevos[
                datos_nuevos["Monto_Solicitado"] <= tope
            ]

            if (
                len(df_val) < 30
                or df_val["Estado_Final"].nunique() < 2
            ):
                print("Muestra insuficiente:", mes, segmento)
                continue

            p = modelo.predict_proba(
                df_val[columnas]
            )[:, 1]

            # mantenemos la proporcion de presupuesto de kaggle
            presupuesto = (
                df_val["Monto_Solicitado"].sum()
                * m18_fraccion_capital
            )

            metricas = evaluar_m18(
                df_val, p, presupuesto
            )

            m18_resultados.append({
                "Modelo": nombre,
                "Mes": mes,
                "Segmento": segmento,
                **metricas
            })

m18_tabla = pd.DataFrame(m18_resultados)
m18_ref_control = pd.DataFrame(m18_control)

print("\nCONTROL DE LA REGRESION ACTUAL")
display(m18_ref_control.round(2))

print(
    "Ganancia promedio:",
    round(m18_ref_control["Ganancia"].mean(), 2)
)
print("Referencia anterior: 509955.80")


In [ ]:

# resumimos los resultados de cada version
m18_resumen = m18_tabla.groupby(
    ["Segmento", "Modelo"], sort=False
).agg(
    ROI_Promedio=("ROI_%", "mean"),
    ROI_Peor_Mes=("ROI_%", "min"),
    Ganancia_Promedio=("Ganancia", "mean"),
    Candidatos_Promedio=("Candidatos", "mean"),
    Monto_Mediano=("Monto_Mediano", "mean"),
    Capital_Promedio=("Capital", "mean"),
    Uso_Presupuesto=("Uso_Presupuesto_%", "mean"),
    Defaults_Promedio=("Defaults", "mean"),
    Sesgo_Prob=("Sesgo_Prob", "mean"),
    Sesgo_Aprobados=("Sesgo_Aprobados", "mean")
).reset_index()

# mostramos las configuraciones de cada segmento
for segmento in m18_segmentos:
    print("\n", segmento)

    display(
        m18_resumen[
            m18_resumen["Segmento"] == segmento
        ].sort_values(
            "ROI_Promedio", ascending=False
        ).round(3)
    )

# vemos la rentabilidad de cada mes
print("\nRENTABILIDAD MENSUAL")

display(
    m18_tabla.pivot_table(
        index=["Segmento", "Modelo"],
        columns="Mes",
        values="ROI_%"
    ).round(2)
)

# revisamos capital, incumplimientos y probabilidades
print("\nDETALLE DE LAS CARTERAS")

display(
    m18_tabla[
        [
            "Mes", "Segmento", "Modelo",
            "Candidatos", "Aprobados", "Capital",
            "Defaults", "Ganancia", "ROI_%",
            "Sesgo_Aprobados"
        ]
    ].round(3)
)


## Modelo 19 — Variables financieras y regularización

**Qué probamos.** Agregamos relaciones financieras y medidas del historial bancario, y comparamos cuatro regresiones: la base, una con variables alternativas, otra entrenada con montos reducidos y esta última con `C=0.1`.

**Qué pasó.** La configuración de montos reducidos y `C=0.1` obtuvo USD 234.026,64 de ganancia mensual promedio y un ROI de 25,579% en octubre, noviembre y diciembre. Superó a la base en los tres meses. En Kaggle, la submission V7 consiguió **USD 130.062 en el leaderboard público**, frente a USD 128.708,58 de nuestra mejor versión anterior.

**Qué decidimos.** Elegimos esta configuración como V7. Es la mejora real que pudimos confirmar en Kaggle, aunque la ventaja local fue moderada.

In [ ]:

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# revisamos los montos y el presupuesto disponible
m19_max_test = float(test["Monto_Solicitado"].max())
m19_demanda_test = float(test["Monto_Solicitado"].sum())
m19_proporcion = min(1.0, 2_000_000 / m19_demanda_test)

print("Monto maximo test:", m19_max_test)
print("Demanda total test:", round(m19_demanda_test, 2))
print("Porcentaje de presupuesto:", round(m19_proporcion * 100, 2))


# armamos las caracteristicas bancarias anteriores a cada solicitud
def preparar_datos_m19(solicitudes, banco):
    sol = solicitudes.copy()
    ban = banco.copy()

    sol["Fecha_dt"] = pd.to_datetime(sol["Fecha_Solicitud"])
    ban["Mes_dt"] = pd.to_datetime(ban["Mes_Referencia"])

    sol["Periodo_Solicitud"] = sol["Fecha_dt"].dt.to_period("M")
    ban["Periodo_Mes"] = ban["Mes_dt"].dt.to_period("M")

    hist = sol[
        ["ID_Solicitud", "ID_Usuario", "Periodo_Solicitud"]
    ].merge(ban, on="ID_Usuario", how="left")

    # dejamos solamente los meses anteriores a la solicitud
    hist = hist[
        hist["Periodo_Mes"] < hist["Periodo_Solicitud"]
    ].copy()

    hist = hist.sort_values(["ID_Solicitud", "Mes_dt"])

    # calculamos los indicadores del historial completo
    generales = hist.groupby("ID_Solicitud").agg(
        Saldo_Promedio=("Saldo_Promedio_Mensual", "mean"),
        Saldo_Minimo=("Saldo_Promedio_Mensual", "min"),
        Atraso_Maximo=("Dias_Atraso_Otras_Deudas", "max"),
        Atraso_Promedio=("Dias_Atraso_Otras_Deudas", "mean"),
        Meses_Observados=("Mes_dt", "count"),
        Meses_Con_Atraso=(
            "Dias_Atraso_Otras_Deudas", lambda x: (x > 0).sum()
        ),
        Meses_Atraso_30=(
            "Dias_Atraso_Otras_Deudas", lambda x: (x >= 30).sum()
        )
    ).reset_index()

    # calculamos los indicadores recientes
    ultimos_3 = hist.groupby("ID_Solicitud").tail(3)

    recientes = ultimos_3.groupby("ID_Solicitud").agg(
        Saldo_Ultimos_3=("Saldo_Promedio_Mensual", "mean"),
        Atraso_Ultimos_3=("Dias_Atraso_Otras_Deudas", "mean")
    ).reset_index()

    ultimo_mes = hist.groupby("ID_Solicitud").tail(1)[
        ["ID_Solicitud", "Saldo_Promedio_Mensual",
         "Dias_Atraso_Otras_Deudas"]
    ].rename(columns={
        "Saldo_Promedio_Mensual": "Saldo_Ultimo",
        "Dias_Atraso_Otras_Deudas": "Atraso_Ultimo"
    })

    # agregamos todas las caracteristicas a las solicitudes
    datos = sol.merge(
        generales, on="ID_Solicitud",
        how="left", validate="one_to_one"
    )

    datos = datos.merge(
        recientes, on="ID_Solicitud",
        how="left", validate="one_to_one"
    )

    datos = datos.merge(
        ultimo_mes, on="ID_Solicitud",
        how="left", validate="one_to_one"
    )

    # calculamos nuevas relaciones entre las variables
    datos["Ratio_Monto_Ingreso"] = (
        datos["Monto_Solicitado"] /
        datos["Ingreso_Mensual"].clip(lower=1)
    )

    datos["Frecuencia_Atraso"] = (
        datos["Meses_Con_Atraso"] /
        datos["Meses_Observados"].clip(lower=1)
    )

    datos["Frecuencia_Atraso_30"] = (
        datos["Meses_Atraso_30"] /
        datos["Meses_Observados"].clip(lower=1)
    )

    datos["Monto_Sobre_Saldo"] = (
        datos["Monto_Solicitado"] /
        (datos["Saldo_Promedio"].clip(lower=0) + 100)
    )

    datos["Saldo_Sobre_Ingreso"] = (
        datos["Saldo_Promedio"] /
        datos["Ingreso_Mensual"].clip(lower=1)
    )

    datos["Ratio_Saldo_Reciente"] = (
        (datos["Saldo_Ultimos_3"] + 100) /
        (datos["Saldo_Promedio"].clip(lower=0) + 100)
    )

    datos["Diferencia_Atraso_Reciente"] = (
        datos["Atraso_Ultimos_3"] - datos["Atraso_Promedio"]
    )

    return datos


# preparamos train y test sin modificar los anteriores
m19_train = preparar_datos_m19(train, bancario)
m19_test = preparar_datos_m19(test, bancario)

assert len(m19_train) == len(train)
assert len(m19_test) == len(test)
assert m19_train["ID_Solicitud"].is_unique
assert m19_test["ID_Solicitud"].is_unique

print("Train:", m19_train.shape)
print("Test:", m19_test.shape)


In [ ]:

# armamos una regresion logistica con las variables elegidas
def crear_logistica_m19(columnas, C=1.0):
    numericas = [
        col for col in columnas
        if col != "Motivo_Prestamo"
    ]

    transformadores = [
        ("numericas", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numericas)
    ]

    if "Motivo_Prestamo" in columnas:
        transformadores.append((
            "motivo", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]), ["Motivo_Prestamo"]
        ))

    return Pipeline([
        ("preprocesador", ColumnTransformer(transformadores)),
        ("modelo", LogisticRegression(
            C=C,
            max_iter=1000,
            random_state=42
        ))
    ])


# evaluamos la cartera usando el valor esperado por dolar
def evaluar_m19(datos, probabilidades):
    cartera = datos[
        ["ID_Solicitud", "Monto_Solicitado",
         "Tasa_Interes", "Estado_Final"]
    ].copy().reset_index(drop=True)

    cartera["Prob_Pago"] = np.asarray(probabilidades)

    cartera["EV_Dolar"] = (
        cartera["Prob_Pago"] *
        (1 + cartera["Tasa_Interes"]) - 1
    )

    presupuesto = (
        cartera["Monto_Solicitado"].sum() * m19_proporcion
    )

    cartera = cartera.sort_values(
        ["EV_Dolar", "ID_Solicitud"],
        ascending=[False, True]
    ).reset_index(drop=True)

    # seleccionamos los prestamos con valor esperado positivo
    capital = 0.0
    indices = []

    for fila in cartera.itertuples():
        if fila.EV_Dolar <= 0:
            continue

        if capital + fila.Monto_Solicitado <= presupuesto:
            indices.append(fila.Index)
            capital += fila.Monto_Solicitado

    aprobados = cartera.iloc[indices]

    # calculamos la ganancia real
    ganancia = np.where(
        aprobados["Estado_Final"] == 1,
        aprobados["Monto_Solicitado"] * aprobados["Tasa_Interes"],
        -aprobados["Monto_Solicitado"]
    ).sum()

    auc = (
        roc_auc_score(
            cartera["Estado_Final"],
            cartera["Prob_Pago"]
        )
        if cartera["Estado_Final"].nunique() == 2
        else np.nan
    )

    return {
        "Candidatos": len(cartera),
        "Aprobados": len(aprobados),
        "Presupuesto": float(presupuesto),
        "Capital": float(capital),
        "Ganancia": float(ganancia),
        "ROI_%": 100 * ganancia / capital if capital else 0,
        "Ganancia_Eq_2M": (
            float(ganancia) * 2_000_000 / presupuesto
            if presupuesto else 0
        ),
        "Uso_Presupuesto_%": (
            100 * capital / presupuesto if presupuesto else 0
        ),
        "Defaults": int(
            (aprobados["Estado_Final"] == 0).sum()
        ),
        "AUC": auc,
        "Sesgo_Prob": float(
            cartera["Prob_Pago"].mean()
            - cartera["Estado_Final"].mean()
        ),
        "Sesgo_Aprobados": float(
            aprobados["Prob_Pago"].mean()
            - aprobados["Estado_Final"].mean()
        ) if len(aprobados) else np.nan
    }


In [ ]:

# definimos las caracteristicas de cada configuracion
m19_base = list(columnas_v4) + ["Motivo_Prestamo"]

m19_alternativas = [
    "Tasa_Interes",
    "Ingreso_Mensual",
    "Ratio_Monto_Ingreso",
    "Saldo_Promedio",
    "Saldo_Minimo",
    "Atraso_Maximo",
    "Atraso_Promedio",
    "Frecuencia_Atraso",
    "Frecuencia_Atraso_30",
    "Atraso_Ultimos_3",
    "Atraso_Ultimo",
    "Monto_Sobre_Saldo",
    "Saldo_Sobre_Ingreso",
    "Ratio_Saldo_Reciente",
    "Diferencia_Atraso_Reciente",
    "Motivo_Prestamo"
]

m19_configuraciones = [
    ("Modelo base", m19_base, False, 1.0),
    ("Variables alternativas", m19_alternativas, False, 1.0),
    ("Montos reducidos", m19_alternativas, True, 1.0),
    ("Montos reducidos C=0.1", m19_alternativas, True, 0.1)
]

m19_resultados = []

# comprobamos las columnas antes de entrenar
for nombre, columnas, _, _ in m19_configuraciones:
    faltantes = set(columnas) - set(m19_train.columns)
    assert not faltantes, f"{nombre}: faltan {faltantes}"

# evaluamos octubre, noviembre y diciembre
for nombre, columnas, filtrar_montos, valor_c in m19_configuraciones:
    for mes in ["2025-10", "2025-11", "2025-12"]:
        inicio = pd.Timestamp(mes + "-01")
        fin = inicio + pd.offsets.MonthBegin(1)

        entrenamiento_original = m19_train[
            m19_train["Fecha_dt"] < inicio
        ].copy()

        entrenamiento = entrenamiento_original.copy()

        # filtramos montos en las ultimas dos configuraciones
        if filtrar_montos:
            entrenamiento = entrenamiento[
                entrenamiento["Monto_Solicitado"] <= m19_max_test
            ]

        # dejamos los usuarios que no aparecen en entrenamiento
        usuarios_vistos = set(
            entrenamiento_original["ID_Usuario"]
        )

        validacion = m19_train[
            (m19_train["Fecha_dt"] >= inicio) &
            (m19_train["Fecha_dt"] < fin) &
            (m19_train["Monto_Solicitado"] <= m19_max_test) &
            (~m19_train["ID_Usuario"].isin(usuarios_vistos))
        ].copy()

        assert len(entrenamiento) > 0

        if len(validacion) < 30:
            raise ValueError(
                f"Muestra insuficiente en {mes}: {len(validacion)}"
            )

        # entrenamos y calculamos las probabilidades
        modelo = crear_logistica_m19(columnas, C=valor_c)

        modelo.fit(
            entrenamiento[columnas],
            entrenamiento["Estado_Final"]
        )

        clases = list(modelo.named_steps["modelo"].classes_)
        indice_pago = clases.index(1)

        probabilidades = modelo.predict_proba(
            validacion[columnas]
        )[:, indice_pago]

        resultado = evaluar_m19(validacion, probabilidades)

        m19_resultados.append({
            "Configuracion": nombre,
            "Mes": mes,
            "Train_Filas": len(entrenamiento),
            **resultado
        })

    print("Terminado:", nombre)

m19_tabla = pd.DataFrame(m19_resultados)

# resumimos las ganancias y el comportamiento de los modelos
m19_resumen = m19_tabla.groupby(
    "Configuracion", sort=False
).agg(
    Ganancia_Promedio=("Ganancia", "mean"),
    Ganancia_Eq_2M=("Ganancia_Eq_2M", "mean"),
    ROI_Promedio=("ROI_%", "mean"),
    ROI_Peor_Mes=("ROI_%", "min"),
    AUC_Promedio=("AUC", "mean"),
    Sesgo_Prob=("Sesgo_Prob", "mean"),
    Sesgo_Aprobados=("Sesgo_Aprobados", "mean"),
    Aprobados_Promedio=("Aprobados", "mean"),
    Defaults_Promedio=("Defaults", "mean"),
    Uso_Presupuesto=("Uso_Presupuesto_%", "mean")
).reset_index()

print("\nRESULTADOS GENERALES")
display(
    m19_resumen.sort_values(
        "ROI_Promedio", ascending=False
    ).round(3)
)

print("\nGANANCIAS POR MES")
display(
    m19_tabla.pivot(
        index="Configuracion",
        columns="Mes",
        values="Ganancia"
    ).round(2)
)

print("\nROI POR MES")
display(
    m19_tabla.pivot(
        index="Configuracion",
        columns="Mes",
        values="ROI_%"
    ).round(2)
)

print("\nDETALLE DE LAS CARTERAS")
display(
    m19_tabla[
        [
            "Configuracion", "Mes", "Candidatos",
            "Aprobados", "Capital", "Defaults",
            "Ganancia", "ROI_%", "Sesgo_Aprobados"
        ]
    ].round(3)
)


In [ ]:

import numpy as np
import pandas as pd

# =====================================================================
# MODELO DEFINITIVO: SUBMISSION V7
# =====================================================================
# Esta fue nuestra mejor versión evaluada en Kaggle (Score: USD 130.062).
# Utiliza regresión logística con regularización (C=0.1), variables financieras
# bancarias, y restringe el entrenamiento al rango de montos del test.

# Seleccionamos las características alternativas definidas previamente
columnas_v7 = list(m19_alternativas)

# Filtramos el entrenamiento para que coincida con la distribución del test
train_v7 = m19_train[
    m19_train["Monto_Solicitado"] <= m19_max_test
].copy()

test_v7 = m19_test.copy()

print("Solicitudes de entrenamiento:", len(train_v7))
print("Solicitudes de test:", len(test_v7))

# Entrenamos la Regresión Logística aplicando el preprocesamiento necesario
modelo_v7 = crear_logistica_m19(
    columnas_v7,
    C=0.1
)

modelo_v7.fit(
    train_v7[columnas_v7],
    train_v7["Estado_Final"]
)

# Obtenemos las probabilidades de pago para el conjunto de test
clases_v7 = list(
    modelo_v7.named_steps["modelo"].classes_
)

indice_pago_v7 = clases_v7.index(1)

probabilidades_v7 = modelo_v7.predict_proba(
    test_v7[columnas_v7]
)[:, indice_pago_v7]

assert np.isfinite(probabilidades_v7).all()
assert ((probabilidades_v7 >= 0) & (probabilidades_v7 <= 1)).all()

# Calculamos la rentabilidad esperada por cada dólar prestado (EV_Dolar)
cartera_v7 = test_v7[
    ["ID_Solicitud", "Monto_Solicitado", "Tasa_Interes"]
].copy()

cartera_v7["Prob_Pago"] = probabilidades_v7

# Fórmula: EV = P(paga) * (1 + Tasa) - 1
cartera_v7["EV_Dolar"] = (
    cartera_v7["Prob_Pago"] *
    (1 + cartera_v7["Tasa_Interes"]) - 1
)

# Ordenamos los préstamos priorizando los más rentables
cartera_v7 = cartera_v7.sort_values(
    ["EV_Dolar", "ID_Solicitud"],
    ascending=[False, True]
).reset_index(drop=True)

cartera_v7["Decision_Aprobacion"] = 0

# Asignamos el presupuesto máximo de USD 2.000.000
presupuesto_v7 = 2_000_000
capital_v7 = 0.0

for indice, fila in cartera_v7.iterrows():
    monto = fila["Monto_Solicitado"]

    # Solo aprobamos si el valor esperado es positivo y hay presupuesto
    if (
        fila["EV_Dolar"] > 0
        and capital_v7 + monto <= presupuesto_v7
    ):
        cartera_v7.loc[
            indice, "Decision_Aprobacion"
        ] = 1

        capital_v7 += monto

# Recuperamos el orden original de las solicitudes para armar el archivo final
submission_v7 = test[["ID_Solicitud"]].merge(
    cartera_v7[["ID_Solicitud", "Decision_Aprobacion"]],
    on="ID_Solicitud",
    how="left",
    validate="one_to_one"
)

# Comprobaciones de seguridad e integridad
assert len(submission_v7) == len(test)
assert submission_v7["ID_Solicitud"].is_unique
assert submission_v7["Decision_Aprobacion"].notna().all()
assert submission_v7["Decision_Aprobacion"].isin([0, 1]).all()

capital_verificado_v7 = (
    submission_v7.merge(
        test[["ID_Solicitud", "Monto_Solicitado"]],
        on="ID_Solicitud",
        validate="one_to_one"
    )
    .eval("Decision_Aprobacion * Monto_Solicitado")
    .sum()
)

assert capital_verificado_v7 <= presupuesto_v7

# Exportamos el archivo CSV para enviar a Kaggle
nombre_archivo_v7 = "submission_v7.csv"

submission_v7["Decision_Aprobacion"] = (
    submission_v7["Decision_Aprobacion"].astype(int)
)

submission_v7.to_csv(nombre_archivo_v7, index=False)

print("\nRESULTADOS DE LA SUBMISSION V7")
print("Aprobados:", int(submission_v7["Decision_Aprobacion"].sum()))
print("Rechazados:", int((submission_v7["Decision_Aprobacion"] == 0).sum()))
print("Capital utilizado: USD", round(capital_verificado_v7, 2))
print("Presupuesto restante: USD", round(presupuesto_v7 - capital_verificado_v7, 2))
print("Probabilidad promedio:", round(float(probabilidades_v7.mean()), 4))
print("Archivo generado:", nombre_archivo_v7)

print("Archivo disponible para descargar:", nombre_archivo_v7)